<div style="background-color: #ADD8E6; border: 1px solid gray; padding: 3px">
    <h3>Code-to-Text Synthetic Data Generation </h3>
    The following is an overview of the workflow:
    <ul>
    <li>Transform the corpus (pdf textbook <i>"Developing MX Applications in ColdFusion"</i>) into chunks of code, using immediately prior sections of text as leading context and generating the following code to text-pairs:
        <ul>
        <li>Code-to-Markdown</li>
        </ul>
    </li>
    <li>Supplies code+context chunks as seed data to a <b>Synthetic Data Generation</b> (<i>sdg_hub</i>) pipeline which generates the following code-to-text pairs for each chunk:
        <ul>
        <li>Code-to-Components</li>
        <li>Code-to-Domain</li>
        <li>Code-to-Summary</li>
        <li>Code-to-Topics</li>
        </ul>
    </li>
    <li>Evaluates and filters the previously generated summaries using LLM-as-Judge metrics:</li>
        <ul>
        <li>Filters out summaries with relevancy score = 0 or faithfulness score = 0 (rubric: 1 for faithful/relevant, 0 for not faithful/not relevant)</li>
        <li>Filters out summaries matching "THIS IS NOT VALID CODE"</li>
        </ul>
    <li>Stores the newly generated code-to-text pairs as a standard HF-compatible dataset.</li>
    </ul>
</div>

In [ ]:
def clone_from_repo(repo_url, destination_path, branch='master'):
    """
    Clones the given git repo to the specified destination.
    """
    ##############################################
    # Imports
    ##############################################
    from git import Repo
    
    try:
        Repo.clone_from(repo_url, destination_path, branch=branch)
        
        print(f"Repository '{repo_url}' cloned successfully to '{destination_path}'.")
    except Exception as e:
        
        print(f"Error cloning repository: {e}")

In [ ]:
def get_processable_files(src, include_extensions=[".pdf"]):
    """
    Returns a list of processable files from the given path.
    """
    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()

    allfiles = []

    for root, _, filenames in os.walk(src):

        for filename in filenames:

            _, extension = os.path.splitext(filename)

            if extension in include_extensions:

                file_path = os.path.join(root, filename).removeprefix(f"{src}/")

                allfiles.append(file_path)

    return allfiles

In [ ]:
def get_chapter_ranges(sourcefilename, do_print=True):
    """
    Returns a list of (beginPage, endPage) ranges for chunks that represent chapters in the given pdf.
    """

    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    import pypdfium2 as pdfium
    
    print("Getting chapter ranges...\n")
    
    pdf = pdfium.PdfDocument(sourcefilename)
    
    ranges = []
    
    begin, end = None, None
    
    for item in pdf.get_toc():
        
        state = "*" if item.n_kids == 0 else "-" if item.is_closed else "+"
        
        target = "?" if item.page_index is None else item.page_index+1
        
        boundary = None
        
        if item.page_index and ((item.n_kids == 0 and item.level < 2) or item.level == 2):
            
            if begin is not None:
                
                end = item.page_index - 1
                
                boundary = [begin, max(begin, end)]
                
                ranges.append(boundary)
                
            begin = item.page_index
            
        if do_print:
            
            if boundary:
                
                print("    " * 2 +  f"(Pages {(boundary[0]+1)} - {(boundary[1]+1)})" + "\n")
                
            print(("    " * item.level) + f"[{state}] {item.title} -> {target}  # {item.view_mode} {item.view_pos}")
            
    return ranges

In [ ]:
def split_chapters(sourcefilename, targetfilename, pagerange):
    """
    Splits the pdf into chapters using the provided page ranges.
    Returns the name of the new pdf chunk.
    """

    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    import pypdfium2 as pdfium
    from pathlib import Path
    
    try:
        
        source_pdf = pdfium.PdfDocument(sourcefilename)
        
        new_pdf = pdfium.PdfDocument.new()
    
        print(f"Retrieving chapter...{targetfilename}, Pages {pagerange[0]} to {pagerange[1]}")
        
        new_page_index = new_pdf.import_pages(source_pdf, pages=list(range(pagerange[0], pagerange[1]+1)))
        
        new_pdf.save(targetfilename)
        
        source_pdf.close()
        
        new_pdf.close()
        
    except Exception as e:
        
        print(f"Error saving {targetfilename}: {e}")

In [ ]:
def convert_to_markdown(sourcefile, markdownfile):
    """
    Converts the file into a markdown file.
    """
    
    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()

    if sourcefile.endswith(".pdf"):
    
        convert_pdf_to_markdown(sourcefile, markdownfile)

    else:
    
        convert_text_to_markdown(sourcefile, markdownfile)
        

def convert_pdf_to_markdown(pdffile, markdownfile):
    """
    Converts the pdf into a markdown file.
    """
    
    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    from docling.document_converter import DocumentConverter
    
    try:
        print(f"Converting {pdffile} to markdown...")
        
        converter = DocumentConverter()
        
        result = converter.convert(pdffile)
        
        markdown_output = result.document.export_to_markdown()

        with open(markdownfile, "w") as file:
            
            file.write(markdown_output)

        print(f"{markdownfile} generated.")
        
    except Exception as e:
        print(f"Error saving {markdownfile}: {e}")
        

def convert_text_to_markdown(textfile, markdownfile):
    """
    Converts the text file into a markdown file.
    """
    
    ##############################################
    # Imports
    ##############################################
    import shutil
    import os
    
    try:
        print(f"Converting {textfile} to markdown...")
    
        os.makedirs(os.path.dirname(markdownfile), exist_ok=True)
    
        shutil.copyfile(textfile, markdownfile)
        
        print(f"File '{markdownfile}' generated.")
        
    except Exception as e:
        print(f"Error saving {markdownfile}: {e}")
    

In [ ]:
def generate_markdown_section_raw_data(file, split_sections=True):
    """
    Generates markdown section chunks from the file.
    """

    ##############################################
    # Imports
    ##############################################
    from datasets import Dataset, Features, Value
    from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
    from langchain.docstore.document import Document
    from sdg_hub.core.blocks import PromptBuilderBlock, LLMChatBlock, LLMParserBlock
    from datasets import Dataset, concatenate_datasets
    import traceback
    import re
    import uuid
    import pprint
    import os

    dataset = None

    def strip_code_section(content):
        """
        Strips out code sections of file.
        """
        code_sections = re.findall(r'([^`]+)```([^`]+)```', content, re.DOTALL | re.MULTILINE)
        
        return code_sections
    
    try:
        print(f"Extracting from markdown {file}...")
        
        filecontent = None
        
        with open(file, mode="r") as f: 
            
            filecontent = f.read()

            dataset = None

            if split_sections and strip_code_section(filecontent):

                print(f"Starting code-to-text mappings for {file}...")

                headers_to_split = [("#", "Header 1"), ("##", "Header 2"),("###", "Header 3")]

                text_splitter = MarkdownHeaderTextSplitter(headers_to_split, strip_headers=False)
            
                splits = text_splitter.split_text(filecontent)
        
                sections = [[strip_code_section(split.page_content) for split in splits if split]][0]

                sections = [(str(uuid.uuid4()), section) for section in sections if section]
                
                dataset = Dataset.from_list([{"code_id": section_id, "code": c, "markdown": s} 
                                              for section_id, section in sections for s, c in section])

            else:

                linesofcode = len(f.readlines())

                if linesofcode > 5_000:

                    raise RuntimeError(f"Files with lines of code > 5000 not currently supported (found {linesofcode} for {file}")

                filepath = file.split(os.sep,1)[1]

                dataset = Dataset.from_list([{"code_id": filepath, "code": f"<!--- {filepath} --->{filecontent}",
                                              "markdown": ""}])

        return dataset        

    except Exception as e:

        print(f"Error occurred while extracting from markdown {file}: {e}")

        traceback.print_exc()

In [ ]:
# ##############################################
# # Generate raw dataset
# ##############################################


def generate_raw_dataset(source_path, target_path, include_extensions=[".pdf"], split_sections=True):
    # ##############################################
    # # Imports
    # ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    from pathlib import Path
    from datasets import Dataset, concatenate_datasets
    
    target_path_chapters = f'{source_path}_chunked_target'
    
    target_path_markdown = f'{source_path}_chunked_markdown'
    
    for directory_path in [source_path, 
                           
                           target_path_chapters, 
                           
                           target_path_markdown,
                          
                           target_path]:
            
        Path(directory_path).mkdir(parents=True, exist_ok=True)
    
    files = get_processable_files(source_path, include_extensions)
    
    dataset = None
    
    for file in files:

        source_path_extension = os.path.splitext(file)[1]

        if split_sections:
        
            ranges = get_chapter_ranges(f"{source_path}/{file}", do_print=False)
        
            for idx, _range in enumerate(ranges):
                
                pdf = f"{target_path_chapters}/{idx}_{file}"
                
                md = f"{target_path_markdown}/{idx}_{file.replace('.pdf', '.md')}"
                
                split_chapters(f"{source_path}/{file}", pdf, _range)
                
                convert_to_markdown(pdf, md)
        
                dataset = generate_markdown_section_raw_data(md) if not dataset else concatenate_datasets([dataset, generate_markdown_section_raw_data(md)])

        else:

            srcfile = f"{source_path}/{file}"
            
            md = f"{target_path_markdown}/{file.replace(source_path_extension, '.md')}"
            
            convert_to_markdown(srcfile, md)
    
            dataset = generate_markdown_section_raw_data(md, split_sections) if not dataset else concatenate_datasets([dataset, generate_markdown_section_raw_data(md, split_sections)])
    
    print("Writing raw dataset to jsonl file...")
    
    return dataset.to_json(f"{target_path}/data.jsonl")

### Run raw_dataset_generation pipeline
Generate the raw dataset!

In [ ]:
# source_path = 'pdf'

# target_path = "json"

# generate_raw_dataset(source_path, target_path, include_extensions=[".pdf"], split_sections=True)

### Generate synthetic data
Generate synthetic code-to-text pairs from the raw dataset using sdg_hub.

In [ ]:
##############################################
# sdg_hub
##############################################

def generate_synthetic_dataset(directory):
    ##############################################
    # Imports
    ##############################################
    from datasets import load_dataset, DatasetDict
    from sdg_hub.core.flow import FlowRegistry, Flow
    import nest_asyncio
    import os
    import traceback
    nest_asyncio.apply()
    
    flow_path = "flows/graphrag_knowledge_generation/flow.yaml"
    
    columns_to_keep = ["code_id", "code", "markdown", "summary", "summary_type", "eval_summary_relevance", "eval_summary_faithfulness"]
    
    flow = Flow.from_yaml(flow_path)
    
    flow.set_model_config(
        model=os.getenv("REFERENCE_LLM_ID"),
        api_base=f"{os.getenv('REFERENCE_LLM_API_BASE')}",
        api_key=os.getenv("REFERENCE_LLM_TOKEN"),
    )
    
    datasets_config = {}
    
    for split in ["train"]:
    
        try:
    
            dataset = load_dataset("json", data_files=f"{directory}/data.jsonl", split=split)
            
            converted_dataset = flow.generate(dataset)
            
            columns_to_remove = [col for col in converted_dataset.column_names if col not in columns_to_keep]
            
            source_dataset = converted_dataset.remove_columns(columns_to_remove)
        
            source_dataset.to_json(f"{directory}/source_data_{split}.jsonl")
        
            datasets_config[split] = source_dataset
    
        except Exception as e:
    
            print(f"Error while generating synthetic data for split {split}: {e}")
    
            traceback.print_exc()
    
    final_dataset = DatasetDict(datasets_config)

    return final_dataset

In [ ]:
# ##############################################
# # Generate synthetic dataset and push to registry
# ##############################################

# sdg_target_path = "json"

# hub_dataset_name = "oaawofolu/emerson"

# final_dataset = generate_synthetic_dataset(sdg_target_path)

# final_dataset.push_to_hub(hub_dataset_name)

In [ ]:
def generate_dataset(git_repo, app_name):
    """Generates a synthetic dataset of code-to-text pairs from the repo and pushes it to HuggingFace."""
    
    hub_dataset_name = f"oaawofolu/cfcode-{app_name}"
    
    source_path = f'cfcode_{app_name}'
    
    target_path = f"cfcode_json_{app_name}"
    
    clone_from_repo(git_repo, source_path)
    
    generate_raw_dataset(source_path, target_path, include_extensions=[".cfm",".cfc", ".cfml", ".java"], split_sections=False)
    
    final_dataset = generate_synthetic_dataset(target_path)
    
    final_dataset.push_to_hub(hub_dataset_name)
    

### Generate synthetic dataset for codebase


In [ ]:
generate_dataset("https://github.com/holtonma/cf_golfap.git", "golfap")
# generate_dataset("https://github.com/kishore31/CheckMate-CMS", "checkmatecms")
# generate_dataset("https://github.com/ehynds/cfml", "ehynds")
# generate_dataset("https://github.com/timblair/cflipsum", "cflipsum")
# generate_dataset("https://github.com/illuminerdi/fusebox_implicit_skelly", "fusebox")